# Busiest Flight Route Analysis

## Problem Statement

You are a data engineer at Air India. The in-flight display team is auditing how much screen space airport names and aircraft model names occupy on boarding screens.

Write a query that returns one row per flight from the flight network.

The solution should use flight, airport, and aircraft information to produce the required output.

---

## Input Tables

### ba_flights

| Column Name | Data Type |
|------------|-----------|
| flight_id | INT |
| origin_airport | VARCHAR |
| destination_airport | VARCHAR |
| plane_id | INT |

### ba_airports

| Column Name | Data Type |
|------------|-----------|
| airport_id | VARCHAR |
| airport_name | VARCHAR |

### ba_planes

| Column Name | Data Type |
|------------|-----------|
| plane_id | INT |
| plane_model | VARCHAR |

---

## Requirements

- Return one row per eligible flight.
- Use airport information associated with both the origin and destination airports.
- Use aircraft information associated with each flight.
- Some airport and aircraft names may contain leading or trailing spaces.
- Character counts must be based on trimmed values.
- Only flights with valid associated airport and aircraft records should be included.
- Return results in ascending order of `flight_id`.

---

## Expected Output Columns

| Column Name | Description |
|------------|-------------|
| flight_id | Unique flight identifier |
| origin_airport_name_length | Length of the origin airport name |
| destination_airport_name_length | Length of the destination airport name |
| plane_model_length | Length of the aircraft model name |

---

## Sample Input

### ba_flights

| flight_id | origin_airport | destination_airport | plane_id |
|-----------|---------------|--------------------|----------|
| 1 | A1 | B1 | 2 |
| 2 | A2 | B2 | 3 |
| 3 | A3 | B3 | 1 |
| 4 | A9 | B1 | 2 |

### ba_airports

| airport_id | airport_name |
|------------|--------------|
| A1 | San Francisco |
| B1 | Los Angeles |
| A2 | New York |
| B2 | Boston |
| A3 | Miami |
| B3 | Orlando |

### ba_planes

| plane_id | plane_model |
|----------|-------------|
| 1 | Airbus A320 |
| 2 | Boeing 737 |
| 3 | Airbus A380 |

---

## Sample Output

| flight_id | origin_airport_name_length | destination_airport_name_length | plane_model_length |
|-----------|---------------------------|---------------------------------|-------------------|
| 1 | 13 | 11 | 10 |
| 2 | 8 | 6 | 11 |
| 3 | 5 | 7 | 11 |

---

## Output Schema

| Column Name | Data Type |
|------------|-----------|
| flight_id | INT |
| origin_airport_name_length | INT |
| destination_airport_name_length | INT |
| plane_model_length | INT |

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import *

# ba_flights
ba_flights_schema = StructType([
    StructField("flight_id", IntegerType(), True),
    StructField("origin_airport", StringType(), True),
    StructField("destination_airport", StringType(), True),
    StructField("plane_id", IntegerType(), True)
])

ba_flights_data = [
    (1, "A1", "B1", 2),
    (2, "A2", "B2", 3),
    (3, "A3", "B3", 1),
    (4, "A9", "B1", 2)
]

ba_flights_df = spark.createDataFrame(
    ba_flights_data,
    schema=ba_flights_schema
)

# ba_airports
ba_airports_schema = StructType([
    StructField("airport_id", StringType(), True),
    StructField("airport_name", StringType(), True)
])

ba_airports_data = [
    ("A1", "San Francisco"),
    ("B1", "Los Angeles"),
    ("A2", "New York"),
    ("B2", "Boston"),
    ("A3", "Miami"),
    ("B3", "Orlando")
]

ba_airports_df = spark.createDataFrame(
    ba_airports_data,
    schema=ba_airports_schema
)

# ba_planes
ba_planes_schema = StructType([
    StructField("plane_id", IntegerType(), True),
    StructField("plane_model", StringType(), True)
])

ba_planes_data = [
    (1, "Airbus A320"),
    (2, "Boeing 737"),
    (3, "Airbus A380")
]

ba_planes_df = spark.createDataFrame(
    ba_planes_data,
    schema=ba_planes_schema
)

In [0]:
flights = ba_flights_df.alias("f")
origin = ba_airports_df.alias("origin")
destination = ba_airports_df.alias("destination")
planes = ba_planes_df.alias("p")

result_df = (
    flights
    .join(
        origin,
        col("f.origin_airport") == col("origin.airport_id"),
        "inner"
    )
    .join(
        destination,
        col("f.destination_airport") == col("destination.airport_id"),
        "inner"
    )
    .join(
        planes,
        col("f.plane_id") == col("p.plane_id"),
        "inner"
    )
    .select(
        col("f.flight_id"),
        length(trim(col("origin.airport_name"))).alias("origin_airport_name_length"),
        length(trim(col("destination.airport_name"))).alias("destination_airport_name_length"),
        length(trim(col("p.plane_model"))).alias("plane_model_length")
    )
)
display(result_df)